# Boston Real Estate Sale Probability - Data Generation

This notebook generates a synthetic dataset of Boston residential properties with calculated sale probabilities. The data simulates various factors that might influence whether a property sells.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)

## Define Boston Neighborhoods

Boston is divided into various neighborhoods, each with its own character and real estate dynamics.

In [ ]:
# Define Boston neighborhoods
boston_neighborhoods = [
    'Back Bay', 'Beacon Hill', 'North End', 'South End', 'Fenway',
    'Allston', 'Brighton', 'Jamaica Plain', 'Roxbury', 'Dorchester',
    'South Boston', 'East Boston', 'Charlestown', 'West Roxbury', 
    'Hyde Park', 'Mattapan', 'Roslindale'
]

## Data Generation Function

The following function creates a synthetic dataset with realistic property characteristics and a calculated sale probability based on various factors.

In [ ]:
# Function to create synthetic Boston real estate data
def generate_boston_real_estate_data(num_samples=500):
    # Current date for reference
    current_date = datetime.now()
    
    # Generate data
    data = {
        # Length of homeownership (years since last purchase)
        'years_owned': np.random.exponential(scale=7, size=num_samples).astype(int) + 1,
        
        # Property characteristics
        'property_value': np.random.normal(loc=750000, scale=300000, size=num_samples),
        'square_feet': np.random.normal(loc=1800, scale=700, size=num_samples),
        'bedrooms': np.random.choice([1, 2, 3, 4, 5], size=num_samples, p=[0.1, 0.25, 0.4, 0.2, 0.05]),
        'bathrooms': np.random.choice([1, 1.5, 2, 2.5, 3, 3.5, 4], size=num_samples, 
                                     p=[0.15, 0.2, 0.3, 0.15, 0.1, 0.05, 0.05]),
        'property_age': np.random.gamma(shape=3, scale=30, size=num_samples) + 5,
        
        # Location characteristics
        'neighborhood': np.random.choice(boston_neighborhoods, size=num_samples),
        'distance_to_t': np.random.exponential(scale=0.7, size=num_samples),
        'school_rating': np.random.normal(loc=7, scale=1.5, size=num_samples),
        
        # Financial characteristics
        'property_tax': np.random.normal(loc=5000, scale=2000, size=num_samples),
        'has_liens': np.random.choice([0, 1], size=num_samples, p=[0.85, 0.15]),
        'lien_amount': np.zeros(num_samples),
        'mortgage_rate': np.random.normal(loc=4.5, scale=1.5, size=num_samples),
        'income_to_mortgage_ratio': np.random.normal(loc=3.5, scale=1.2, size=num_samples),
        
        # Market conditions
        'market_inventory_months': np.random.normal(loc=3.5, scale=1.5, size=num_samples),
        'avg_days_on_market': np.random.gamma(shape=2, scale=15, size=num_samples) + 10,
    }
    
    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    # Clean up and transform data
    df['property_value'] = np.maximum(200000, df['property_value']).astype(int)
    df['square_feet'] = np.maximum(400, df['square_feet']).astype(int)
    df['property_age'] = np.maximum(0, df['property_age']).astype(int)
    df['school_rating'] = np.clip(df['school_rating'], 1, 10).round(1)
    df['property_tax'] = np.maximum(1000, df['property_tax']).astype(int)
    df['distance_to_t'] = np.round(df['distance_to_t'], 2)
    df['mortgage_rate'] = np.clip(df['mortgage_rate'], 2, 8).round(2)
    df['income_to_mortgage_ratio'] = np.maximum(0.5, df['income_to_mortgage_ratio']).round(2)
    df['market_inventory_months'] = np.maximum(0.5, df['market_inventory_months']).round(1)
    df['avg_days_on_market'] = np.maximum(1, df['avg_days_on_market']).astype(int)
    
    # Create lien amount for properties with liens
    df.loc[df['has_liens'] == 1, 'lien_amount'] = np.random.gamma(
        shape=2, scale=5000, size=df['has_liens'].sum())
    df['lien_amount'] = df['lien_amount'].astype(int)
    
    # Create last transaction date based on years owned
    df['last_transaction_date'] = df['years_owned'].apply(
        lambda x: (current_date - timedelta(days=365 * x)).strftime('%Y-%m-%d'))
    
    # Calculate the probability of selling
    # This is a simplified model that combines several factors
    df['sale_probability'] = (
        # Base probability decreases with length of ownership but peaks around 5-7 years
        0.4 * np.exp(-((df['years_owned'] - 6) ** 2) / 50) +
        
        # NEW: Elderly homeowner effect - increased probability after 25 years of ownership
        0.25 * np.clip((df['years_owned'] - 25) / 5, 0, 1) +
        
        # Liens decrease probability of selling
        -0.2 * (df['has_liens']) +
        
        # Financial factors
        -0.1 * np.clip((df['property_tax'] - 5000) / 10000, -0.1, 0.1) +
        0.1 * np.clip((df['income_to_mortgage_ratio'] - 3) / 5, -0.1, 0.1) +
        
        # Market conditions
        -0.1 * np.clip((df['market_inventory_months'] - 3) / 5, -0.1, 0.1) +
        -0.05 * np.clip((df['avg_days_on_market'] - 30) / 60, -0.05, 0.05) +
        
        # Neighborhood factor (random effect)
        0.1 * np.random.normal(0, 1, num_samples)
    )
    
    # Clip probability between 0 and 1
    df['sale_probability'] = np.clip(df['sale_probability'], 0.01, 0.99).round(2)
    
    return df

## Generate the Dataset

Now we'll generate 1,000 Boston properties with all the necessary characteristics.

In [ ]:
# Generate the dataset
boston_real_estate = generate_boston_real_estate_data(1000)

# Display the first few rows
print("Sample Boston Real Estate Dataset:")
boston_real_estate.head()

## Save the Dataset

Let's save the dataset to a CSV file for use in subsequent analysis.

In [ ]:
# Save to CSV
boston_real_estate.to_csv('../data/boston_real_estate_sale_probability.csv', index=False)
print("Dataset saved to '../data/boston_real_estate_sale_probability.csv'")

## Data Generation Summary

We've created a synthetic dataset with the following characteristics:

- 1,000 Boston properties
- 17 features plus target variable (sale probability)
- Realistic distributions for property values, sizes, and ages
- Modeled relationships between features and sale probability
- Non-linear effects (especially for years_owned)

This dataset will be used in subsequent notebooks for exploratory analysis and predictive modeling.